In [ ]:
#!pip install geopandas

In [ ]:
import geopandas as gpd
import pandas as pd
import folium
pd.set_option("display.max_columns", 20)
#pd.set_option("display.max_rows", 1000)

In [ ]:
zones= gpd.read_file('taxi_zones/taxi_zones.shp')

In [ ]:
zones.shape

In [ ]:
zones.head()

In [ ]:
print(zones.crs)  # Projected System measured in metres

- `LocationID`: is the Taxizone ID
- `borough`: entire city
- `geometry`: essential part to pinpoint the location in a map
- `zone`: neighbourhood

All the locations of the Earth can be located by (Latitude, Longitude).
- Longitude: It basically tells how far is the place East/West of the `Prime Meridian`.
- Latitude: It basically tells how far is the place North/South of the `Equator`.

New York City:
1. Latitude:   40.7° North
2. Longitude:  74.0° West

This means New York is:

- 40.7 degrees north of the Equator
- 74 degrees west of the Prime Meridian

### Projected CRS (Coordinate Reference Systems):

It is a way of turning the Earth into a flat map that a computer can measure easily. 

By comparison **Geographic CRS**  indicates where a place on Earth is by:
1. Latitude
2. Longitude
3. Degrees

A Projected CRS uses **X** (left/right, usually East/West) and **Y-axis** (means up and down, usually North/South). Why do we need Projected CRS?

Latitude and Longitude are measured in degrees, so calculating an area directly is difficult. A Projected CRS converts the coordinates into units such as:

- metres
- feet

Thus, making it easier to calculate distance, area, length or nearby locations.

In [ ]:
# Convert the Projected CRS to Geographic CRS:
# to_crs(): Transform geometries to a new coordinate reference system.

'''
EPSG: 4326 is a Geographic CRS (Longitude and latitude in degrees)

'''
zones= zones.to_crs(epsg= 4326)
zones.head()

In [ ]:
print(zones.crs)

A `Polygon` is one continuous, closed area. Imagine drawing a boundary around one park:

      __________
     /          \
    /    Park    \
    \            /
     \__________/

The `Closed Polygon` are a list of `(Latitude, Longitude)`.

Examples of Polygon features:

- one city district
- one park
- one lake
- one building
- one taxi zone
- one country with one continuous land area

A `MutliPolygon` is made of two or more separate polygons. Example would be an area having two islands, these pieces are physically disconnected, but they belong to the same feature. Thus, the are stored in MultiPolygon.

Examples include:

- a country containing several islands
- a district divided by water
- a taxi zone containing disconnected land or water areas
- an administrative area consisting of multiple separate pieces
     


In [ ]:
zones["geometry"][0]

In [ ]:
zones["geometry"][1]  # Multipolygon

### Analyzing Busiest Pickup Zones:

In [ ]:
df= pd.read_csv("uber-raw-data-sep14.csv")
df.head()

In [ ]:
# converting lat and lon to geometry:
#! pip install shapely

A **GeoDataFrame** object is a pandas.DataFrame that has one or more columns
containing geometry. In addition to the standard DataFrame constructor arguments,
GeoDataFrame also accepts the following keyword arguments:

##### Parameters
----------
- crs : value (optional)
    Coordinate Reference System of the geometry objects. Can be anything accepted by
    :meth:`pyproj.CRS.from_user_input() <pyproj.crs.CRS.from_user_input>`,
    such as an authority string (eg "EPSG:4326") or a WKT string.
- geometry : str or array-like (optional)
    Value to use as the active geometry column.
    If str, treated as column name to use. If array-like, it will be
    added as new column named 'geometry' on the GeoDataFrame and set as the
    active geometry column.


**points_from_xy()**:
Generate GeometryArray of shapely Point geometries from x, y(, z) coordinates.
In case of geographic coordinates, it is assumed that longitude is captured by
``x`` coordinates and latitude by ``y``.

##### Parameters
----------
- x, y, z : iterable
- crs : value, optional
    Coordinate Reference System of the geometry objects. Can be anything accepted by
    :meth:`pyproj.CRS.from_user_input() <pyproj.crs.CRS.from_user_input>`,
    such as an authority string (eg "EPSG:4326") or a WKT string.


----------------------------------

Why convert the Latitude and Longitude to point geometry?

- Latitude and Longitude tells us where the location is, but `Point geometry` gives `GeoPandas` an object on which it can perform spatial operations.

The Point geometry tells GeoPandas:

- this is one map location;
- longitude is its horizontal position;
- latitude is its vertical position;
- use this location in geographical calculations.

In [ ]:
# Convert Coordinates into point geometries:
uber_geo= gpd.GeoDataFrame(df, geometry= gpd.points_from_xy(df['Lon'], df['Lat']), crs= "EPSG:4326")
uber_geo.head()

In [ ]:
zones.columns

**Spatial Joins** matches rows using their positions on a map based on the spatial relationship between their geometries.


In [ ]:
uber_zones= gpd.sjoin(uber_geo, zones[['zone', 'LocationID', 'borough',
       'geometry']], how= "left", predicate= "within")

uber_zones.head()

In [ ]:
zones[zones['zone']== "West Chelsea/Hudson Yards"]

`index_right` after the `spatial_join` is the original row index from the right-hand polygon GeoDataFrame.

In [ ]:
uber_zones.shape

In [ ]:
uber_zones['zone'].nunique()  # 254 unique zones

In [ ]:
uber_zones['zone'].value_counts() # Returns frequency of unique values in a column orderd by the frequency of the number of occurrences

In [ ]:
# Display top 15 busiest pick-up zones:
uber_zones['zone'].value_counts()[0:15]

In [ ]:
uber_zones.columns

In [ ]:
uber_zones['borough'].value_counts()  

Manhattan area gets more uber pickups.

##### How many pickups occurred in every zone, separated by borough?

In [ ]:
# Top 5 zones from each borough:

top_pickup_zones= (uber_zones.groupby(['borough', 'zone'])
    .size()
    .reset_index(name= "pickup_count")
    .sort_values(["borough", "pickup_count"], ascending= [True, False])
    .groupby("borough")
    .head(5))

print(top_pickup_zones)

### Creating Bubble Map:

In [ ]:
# Filtering top 20 pickup zones and storing it as a DataFrame:

uber_pickup_zones=(uber_zones['zone'].value_counts()[0:20]).reset_index()
uber_pickup_zones.columns= ['zone', 'pickup_count']
uber_pickup_zones

In [ ]:
zone_bubble_df= zones.merge(uber_pickup_zones, on= 'zone', how= 'inner') # keep all those data that matches both the table based on zone
zone_bubble_df.head(3)

In [ ]:
# Convert the `geometry` to projected CRS: to get the centroid values from a flat map

zone_project_csr= zone_bubble_df.to_crs(epsg= 2263)
zone_project_csr.head(3)

The **centroid** calculates a representative central point for the polygon. That point becomes the bubble’s position.

In [ ]:
zone_bubble_df['centroid']= (zone_project_csr['geometry'].centroid).to_crs(epsg= 4326)

zone_bubble_df.head(3)

In [ ]:
base_map = folium.Map(
    location=[40.75, -73.97],
    zoom_start=12
)

In [ ]:
for index, row in zone_bubble_df.iterrows():
    folium.CircleMarker(
        location= [row['centroid'].y, row['centroid'].x],
        radius=row['pickup_count']/2000,
        color= "crimson",
        fill= True,
        fill_opacity= 0.6,
        tooltip= (f'{row["zone"]}: {int(row["pickup_count"])} pickups')
    ).add_to(base_map)

In [ ]:
base_map

### Marker Cluster Analysis

1. Rounding Lat and Lon to three decimal places to have a nearby location.
2. Custom Data Binning
3. FastMarkerCluster lib from folium.plugins

In [ ]:
uber_geo.head(3)

In [ ]:
uber_geo['Lat_bin']= uber_geo['Lat'].round(3)
uber_geo['Lon_bin']= uber_geo['Lon'].round(3)

In [ ]:
# groupping by the lat and lon binning:
df_binning= uber_geo.groupby(['Lat_bin', 'Lon_bin'], as_index= False).size().sort_values(by= "size", ascending= False)
df_binning.head()

In [ ]:
from folium.plugins import FastMarkerCluster

In [ ]:
basemap_cluster = folium.Map(
    location=[40.75, -73.97],
    zoom_start=10
)

In [ ]:
# Marker Cluster
FastMarkerCluster(df_binning[['Lat_bin', 'Lon_bin', 'size']]).add_to(basemap_cluster)

In [ ]:
basemap_cluster

In conclusion, Marker Clustering is a map based visualization and `data grouping technique` that merges nearby geographic points into dynamic cluster icons. It reduces dense spatial data into numbered circles that expand and collapse smoothly as the user zooms in and out.

### Uber demand over time:

To visualize the daily uber demands over time in New York city we need to have:
- Latitude
- Longitude
- Time on that particular day

In [ ]:
uber_geo.dtypes

In [ ]:
uber_geo['Date/Time']= pd.to_datetime(uber_geo['Date/Time'], errors= 'coerce')
uber_geo['Date/Time']

In [ ]:
# .dt.floor("h") moves every timestamp to the beginning of its hour. 
# Pandas provides dt.floor() specifically for rounding datetime values down to a selected frequency.
uber_geo['time_bin']= uber_geo['Date/Time'].dt.floor("h")

In [ ]:
uber_geo.head()

In [ ]:
uber_geo.tail()

In [ ]:
uber_geo.loc[0, "time_bin"]

In [ ]:
grouped_pickups= (uber_geo
    .groupby(['time_bin', 'Lat_bin', 'Lon_bin'], as_index= False)
    .size()
    .rename(columns={'size': 'pickup_counts'}))

In [ ]:
grouped_pickups

In [ ]:
timeline= sorted(grouped_pickups["time_bin"].unique())
timeline

In [ ]:
heatmap_data=[]
for timestamp in timeline:
    time_df= grouped_pickups[grouped_pickups['time_bin']== timestamp]
    frame= time_df[
        ['Lat_bin', 'Lon_bin', 'pickup_counts']
    ].values.tolist()
    heatmap_data.append(frame)

In [ ]:
heatmap_data

In [ ]:
from folium.plugins import HeatMapWithTime

In [ ]:
basemap_demand = folium.Map(
    location=[40.75, -73.97],
    zoom_start=11
)

In [ ]:
time_labels= [pd.Timestamp(timestamp).strftime("%Y-%m-%d %H:%M")for timestamp in timeline]


In [ ]:
HeatMapWithTime(
    data=heatmap_data,
    index=time_labels,
    radius=12,
    auto_play=True
).add_to(basemap_demand)

In [ ]:
basemap_demand

In [ ]:
basemap_demand.save(r'plots/uber_demand.html')